# <center> Clase 10: NLP mediante transformers</center>



**Agenda:**

    10.1.- Visión general del modelo transformer
        10.1.1.- Arquitectura del transformer
        10.1.2.- Librerías y modelos disponibles

    10.2.- Modelos basados en transformers
        10.2.1.- Primeros modelos de transformers
        10.2.2.- Nuevos modelos LLM
        10.2.3.- Sistemas de automatización

    10.3.- Ejemplos de uso
        10.3.1.- Clasificación
        10.3.2.- Reconocimiento de entidades
        10.3.3.- Sumarización
        10.3.4.- Pregunta - Respuesta
        10.3.5.- Generación de texto
        10.3.6.- LangChain
        10.3.7.- Auto-GPT
     

## 10.1.- Visión general del modelo transformer

### 10.1.1.- Arquitectura del transformer

La arquitectura de Transformer sigue una estructura de codificador-decodificador, pero no depende de la recurrencia y las convoluciones para generar una salida. En cada paso, como el modelo es autorregresivo, va consumiendo los símbolos generados previamente como entrada adicional al generar el siguiente.

**Codificador**

La tarea del codificador, en la mitad izquierda de la arquitectura Transformer, es mapear una secuencia de entrada a una secuencia de representaciones continuas, que luego se alimenta a un decodificador.

El codificador consta de una pila de 6 capas idénticas, donde cada capa se compone de dos subcapas:
* La **primera subcapa** implementa un mecanismo de autoatención de varios cabezales. Has visto que el mecanismo multicabezal implementa cabezales que reciben una versión proyectada linealmente (diferente) de las consultas, claves y valores, cada uno para producir salidas en paralelo que luego se utilizan para generar un resultado final.
* La **segunda subcapa** es una red feed-forward totalmente conectada que consta de dos transformaciones lineales con activación ReLU en el medio.
Las seis capas del codificador aplican las mismas transformaciones lineales a todas las palabras en la secuencia de entrada, pero cada capa emplea un peso diferente. Además, cada una de estas dos subcapas tiene una conexión residual a su alrededor y es sucedida por una capa de normalización que normaliza la suma calculada entre la entrada de la subcapa y la salida generada por la propia subcapa.

Una consideración importante a tener en cuenta es que la arquitectura de Transformer no puede capturar inherentemente ninguna información sobre las posiciones relativas de las palabras en la secuencia, ya que no utiliza la recurrencia. Esta información debe inyectarse introduciendo codificaciones posicionales en las incrustaciones de entrada mediante una técnica llamada **positional encoding** que, mediante diversas transformaciones cíclicas de diferente periodicidad (típicamente trigonométricas) transfiere esta información al resto de la red. De este modo, aunque prima la relación entre palabras, difiere de los modelos BoW principalmente en que combina aquella información con esta información de ordenación de las palabras. Los vectores de codificación posicional tienen la misma dimensión que las incrustaciones de entrada y se generan utilizando funciones de seno y coseno de diferentes frecuencias. Luego, simplemente se suman a las incrustaciones de entrada para inyectar la información posicional.

**Decodificador**

El decodificador, en la mitad derecha de la arquitectura, recibe la salida del codificador junto con la salida del decodificador en el paso de tiempo anterior para generar una secuencia de salida.

El decodificador comparte varias similitudes con el codificador, pues también consta de una pila de 6 capas idénticas que se componen cada una de tres subcapas:

* La **primera subcapa** recibe la salida anterior de la pila del decodificador, la aumenta con información posicional e implementa la autoatención de varios cabezales sobre ella. Mientras que el codificador está diseñado para prestar atención a todas las palabras en la secuencia de entrada, independientemente de su posición en la secuencia, el decodificador se modifica para prestar atención únicamente a las palabras anteriores. Por lo tanto, la predicción de una palabra en la posición solo puede depender de las salidas conocidas para las palabras que le preceden en la secuencia. En el mecanismo de atención de múltiples cabezas (que implementa múltiples funciones de atención única en paralelo), esto se logra introduciendo una máscara sobre los valores producidos por la multiplicación escalada de matrices. Este enmascaramiento se implementa suprimiendo los valores de la matriz que de otro modo corresponderían a conexiones ilegales. Este enmascaramiento hace que el decodificador sea unidireccional (a diferencia del codificador bidireccional).

* La **segunda capa** implementa un mecanismo de autoatención de cabezales múltiples similar al implementado en la primera subcapa del codificador. En el lado del decodificador, este mecanismo de cabezales múltiples recibe las consultas de la subcapa del decodificador anterior y las claves y valores de la salida del codificador. Esto permite que el decodificador preste atención a todas las palabras en la secuencia de entrada.

* La **tercera capa** implementa una red feed-forward completamente conectada, similar a la implementada en la segunda subcapa del codificador.

Las tres subcapas en el lado del decodificador también tienen conexiones residuales a su alrededor y están reemplazadas por una capa de normalización. Las codificaciones posicionales (positional encoding) también se agregan a las incorporaciones de entrada del decodificador de la misma manera que se explicó anteriormente para el codificador.

### 10.1.2.- Librerías y modelos disponibles

Aunque los primeros modelos de aprendizaje profundo supusieron una enorme revolución entre 2012 y 2017, sería a partir de los transformers cuando el procesamiento del lenguaje natural conocería su mejor momento.

Aunque inicialmente se empezaron a utilizar librerías como ktrain para tratar modelos de transformers, hoy en día la librería más completa y potente es la librería **transformers, de Hugging Face**. Entre los modelos más comunes que se pueden descargar, fine-tunear y explotar destacan los siguientes:
* **Bidirectional Encoder Representations from Transformers (BERT)**: no sólo se refiere a la arquitectura de un modelo, sino a un modelo entrenado en sí mismo, que puede descargar y utilizar gratuitamente. Fue entrenado por investigadores de Google en un corpus de texto masivo y se ha convertido en una especie de navaja de uso general para la PNL. Se puede ampliar para resolver un montón de tareas diferentes, como resumen de textos, respuesta a preguntas, clasificación, reconocimiento de entidades, similitud de textos, detección de mensajes ofensivos/profanos, entender las consultas de los usuarios, etc.
* **Generated Pre-trained Transformer (GPT-x)**: GPT-3, el útimo modelo de GPT, consiste en una red de transformadores con un tamaño sin precedentes de un contexto de 2048 tokens y 175 mil millones de parámetros (que requieren 800 GB de almacenamiento). El método de entrenamiento es "preentrenamiento generativo", lo que significa que está entrenado para predecir cuál es el próximo token. El modelo demostró un fuerte aprendizaje de pocos disparos en muchas tareas basadas en texto. Fue creado por OpenAI, un laboratorio de investigación de inteligencia artificial con sede en San Francisco. La calidad del texto generado por GPT-3 es tan alta que puede ser difícil determinar si fue escrito por un ser humano o no, lo que tiene beneficios y riesgos.
* **Reformer**: pretende resolver las mismas tareas que los anteriores transformers pero reduciendo el número de parámetros y el tiempo de entrenamiento. Para ello, por un lado, se reemplaza la atención del producto escalar por una que usa LSH (Locality Sensitive Hashing), por otro lado, se usan capas residuales reversibles en lugar de los residuos estándar, lo que permite almacenar activaciones solo una vez en el proceso de entrenamiento en lugar de una por capa. El modelo resultante, el Reformer, se desempeña a la par de los modelos de Transformer y es mucho más eficiente en memoria y mucho más rápido en secuencias largas.
* **Pattern-Exploiting Training (PET)**: en entornos de bajos recursos, PET e iPET superan significativamente el entrenamiento supervisado regular, varias líneas de base semisupervisadas e incluso GPT-3 a pesar de requerir muchos menos de parámetros. La variante iterativa de PET (iPET) entrena múltiples generaciones de modelos e incluso se puede usar sin ningún dato supervisión para el entrenamiento.
* **Text-to-Text Transformer (T5)**: modelo de codificador-decodificador entrenado previamente en una combinación de múltiples tareas supervisadas y no supervisadas y para el cual cada tarea se convierte en un formato de texto a texto vía transfer learning. T5 funciona bien en una variedad de tareas listas para usar al anteponer un prefijo diferente a la entrada correspondiente a cada tarea, por ejemplo, para traducción.

Aparte de estos, existen modelos de transformers cuyo código no ha sido abierto y que pueden ser invocados como servicio a través de una API, como las últimas versiones de GPT-3 en la actualidad o el nuevo ChatGPT, basado en GPT-3.5.
Además, durante 2022, otros muchos modelos de lenguaje basados en transformers han ido abriéndose paso, como PaLM, que se aplicaría a actividades matemáticas mediante Minerva, o BLOOM, versión open source disponible en la plataforma de Hugging Face.

### 10.1.3.- Nuevas aplicaciones empresariales

La irrupción de ChatGPT basado en los modelos de lenguaje con transformers ha supuesto una revolución de potenciales aplicaciones empresariales, así como movimientos de censura por parte de actividades que podrían quedar afectadas por esta nueva tecnología. Los principales efectos que se esperan para los venideros serían los siguientes:
* **Enseñanza**: los alumnos podrían no requerir de el mismo nivel de atención por parte de un profesor, pues podrían tener acceso instantáneao a la respuesta de muchas de sus preguntas. Esto podría verse como una amenaza por parte de los profesores, además de que no resultaría sencillo distinguir en qué casos los trabajos a distancia han sido realizados por el alumno o un sistema de inteligencia artificial.
* **Sistemas expertos**: actividades de alta especialización como la medicina, la asesoría legal o la ingeniería podrían verse afectadas por este tipo de asistentes virtuales, o al menos reducir la carga de trabajo de estos profesionales.
* **Programación**: se ha podido comprobar que ChatGPT es bastante bueno programando y se esperaría que pudiera reemplazar a programadores en un futuro cercano. Aquí la pregunta que surge sería ¿podría ser ChatGPT capaz de programarse a sí mismo?
* **Actividades artísticas**: ChatGPT es capaz de narrar historias con gran variedad de estilos e imaginación, al igual que, apoyándose en otros modelos de transformers para imagen o audio es capaz de componer música o de crear imágenes sorprendentes para cuadros. Bien sea como asistente al artista o como reemplazo de estos, parece que ésta será una de las principales afectadas por estas tecnologías.
* **Trabajos de gestión automatizables**: ya estaban siendo automatizados a través de RPA, pero se esperaria que con el empuje de estos sistemas se acelere este proceso y que todas estas actividades pueden ser completamente automatizadas antes de lo esperado.
* **Actividades de supervisión y control**: a diferencia de las personas, que no están exentos de cansancio y de distracciones, los nuevos sistemas de inteligencia artificial pueden reforzar estas tareas y bien sea apoyándose en humanos para confirmar situaciones ambiguas, bien sea resolviéndolo directamente cuando la fiabilidad supere un cierto umbral de calidad.

Muchas otras actividades podría ser impactadas por la irrupción de estas tecnologías, aunque las que parece que serían más difíciles de automatizar serían aquellas en las que no existe un guión claro a seguir, actividades físicas o de gestión de personas, resolución de conflictos y todo tipo de situaciones inesperadas e irrepetibles.

## 10.2.- Modelos basados en transformers

### 10.2.1.- Primeros modelos de transformers

#### Modelos BERT (Google)

Existen varias versiones de BERT para diferentes propósitos, entre los que destacan los siguientes:
* **XLNet**: transformer bidireccional que utiliza una metodología de entrenamiento mejorada, datos más grandes y más poder computacional para lograr mejores métricas de predicción que BERT en tareas de 20 idiomas.
* **RoBERTa**: presentado en Facebook, utiliza un enfoque sólidamente optimizado, con un 1000% más de datos y potencia de cómputo.
* **DistilBERT**: aprende una versión destilada, conservando el 97% de rendimiento pero usando solo la mitad de la cantidad de parámetros. La técnica "destilación" consiste en aproximar al algoritmo original de Google, de modo que se sustituye la gran red neuronal por una más pequeña.
* **ALBERT**: se publicó aproximadamente al mismo tiempo que DistilBERT. Si bien ALBERT también reduce el tamaño del modelo de BERT, no tiene que sacrificar el rendimiento. En comparación con DistilBERT, que usa BERT como maestro para su proceso de destilación, ALBERT se entrena desde cero (al igual que BERT).
* **BART** aunque su rendimiento en las tareas de comprensión del lenguaje natural es comparable al de Roberta, BART admite tareas que involucran la generación de lenguaje natural (por ejemplo, la tarea de resumen abstracto, que requiere generar texto nuevo para resumir un texto de entrada largo, en contraste con la tarea de resumen tradicional que se extraen algunos segmentos del texto de entrada para hacer el resumen).
* **MobileBERT**: marca un hito en la misión de permitir que los modelos de lenguaje preentrenados funcionen en pequeños dispositivos móviles. En comparación con BERT, MobileBERT es 4,3 veces más pequeño, 5,5 veces más rápido, pero sigue teniendo un rendimiento comparable.
* **ELECTRA**: introduce un nuevo objetivo de formación previa que hace que el proceso de formación previa sea mucho más eficaz y eficiente. El modelo ahora aprende de todos los tokens de entrada (a diferencia de modelos anteriores como BERT en los que solo el 15% de los tokens tienen efectos en el aprendizaje de parámetros del modelo). Iguala el rendimiento de RoBERTa y XLNet con poco menos del 25% del tiempo de cálculo, los supera (e incluso a ALBERT) en la mayoría de las tareas de GLUE y SQuAD si se usa la misma cantidad de cómputo.

#### Modelos GPT (OpenAI)

Los modelos de transformadores generativos preentrenados (GPT) de OpenAI han tomado por asalto a la comunidad de procesamiento de lenguaje natural al introducir modelos de lenguaje muy poderosos. Estos modelos pueden realizar varias tareas de NLP como responder preguntas, vinculación textual, resumen de texto, etc. sin ningún entrenamiento supervisado. Estos modelos de lenguaje necesitan muy pocos o ningún ejemplo para comprender las tareas y desempeñarse de manera equivalente o incluso mejor que los modelos de última generación entrenados de manera supervisada.

Aunque GPT-2 está disponible de forma gratuita, **GPT-3** se accede a través de los end-points habilitados por OpenAI. Con GPT-3, el número de parámetros ha alcanzado el orden de cientos de billones (americanos) de parámetros, siendo además evolucionado actualmente a GPT-3.5 (utilizado en chatGPT). **ChatGPT** (OpenAI, 2022) es un sistema de conversación compuesto de un gran modelo de lenguaje ajustado con técnicas de aprendizaje <ins>tanto supervisadas como por refuerzo</ins>. Además de responder a la pregunta, esta AI tiene un sentido del contexto y reconoce todo lo que habéis estado hablando hasta ahora, por lo que si le haces alguna pregunta relacionada con una respuesta que te ha dado, sabrá identificar si te refieres a ello sin tener que darle toda la explicación. ChatGPT se basa en el modelo GPT-3.5 de OpenAI, una versión mejorada de GPT-3 y, en comparación con su predecesor (InstructGPT), evita las respuestas ofensivas con el fin de evitar conflictos con los usuarios, las consultas se filtran a través de una API de moderación y <ins>se descartan las indicaciones potencialmente racistas o sexistas</ins>.

Los modelos fueron entrenados en colaboración con Microsoft en su infraestructura de Azure y funciona para múltiples idiomas. Para ello utilizaron active learning, una técnica en de interacción hombre-máquina, de modo que los humanos le enseñaban a seguir instrucciones, mantenerse sincero y no ser parcial, ni tóxico ni dañino. Mediante Reinforcement Learning from Human Feedback (**RLHF**), utilizando humanos como señal de recompensa para afinar nuestros modelos y luego se recopilaron conjuntos de datos de demostraciones escritas por humanos del comportamiento de salida deseado en solicitudes enviadas a la API de OpenAI para entrenar líneas base de aprendizaje supervisado mediante imitation learning. Por último, se recopilaron un conjunto de datos de etiquetas humanas para entrenar un modelo de recompensas usando el algoritmo de reinforcement learning Proximal Policy Optimization (PPO).

El último hito fue el lanzamiento en 2023 de la nueva versión **GPT-4**, dando un salto en sus capacidades y adquiriendo la posibilidad de analizar tanto texto como imágenes. Esta mejor se ha visto de forma palpable en diferentes pruebas, entre las que destacó la evaluación en diferentes exámenes universitarios y test de programación, en los cuales GPT-4 se situó entre las mejores calificaciones, hito del que estaba lejos de llegar su antecesor, ChatGPT.

### 10.2.2.- Nuevo ecosistema LLM

Los nuevos **Large Language Models (LLM)** irrumpieron al escenario mundial con el lanzamiento de GPT-3 de OpenAI en 2020, creciendo en popularidad desde entonces. El interés en los LLM y la disciplina más amplia de la IA generativa se ha disparado. Las razones de esto son probablemente el impulso ascendente continuo de los avances significativos en los LLM. Uno de los primeros hitos fue el chatbot LaMDA "consciente" de Google. Se lanzó el primer LLM de alto rendimiento y código abierto llamado BLOOM. OpenAI lanzó su modelo de incrustación de texto de próxima generación y la próxima generación de modelos "GPT-3.5". Después de todos estos grandes avances en el espacio LLM, OpenAI lanzó ChatGPT, lo que puso a los LLM en el centro de atención.

**LangChain** apareció hace apenas un año. Su creador, Harrison Chase, hizo el primer compromiso a fines de octubre de 2022. Dejando un par de meses de desarrollo antes de quedar atrapado en la ola LLM. A pesar de ser los primeros días de la biblioteca, ya está repleta de características increíbles para crear herramientas increíbles en torno al núcleo de los LLM. En este artículo, presentaremos la biblioteca y comenzaremos con el componente más sencillo que ofrece LangChain: LLM.

Otro importante hito fue el lanzamiento de **Auto-GPT**, un sistema para secuenciar llamadas a GPT-4 e interactuar con aplicaciones a partir del motor de inteligencia de OpenAI (o de otros modelos similares actuales). Mediante Auto-GPT se están consiguiendo llevar a cabo tareas complejas y automatizar gran cantidad de actividades hasta ahora sólo desarrolladas por humanos.

## 10.3.- Ejemplos de uso

### 10.3.1.- Clasificación

In [1]:
# from https://colab.research.google.com/github/huggingface/notebooks/blob/master/examples/text_classification.ipynb

### 10.3.2.- Reconocimiento de entidades

In [2]:
# from https://colab.research.google.com/github/huggingface/notebooks/blob/master/examples/token_classification.ipynb

### 10.3.3.- Sumarización

In [ ]:
# from https://colab.research.google.com/github/huggingface/notebooks/blob/master/examples/summarization.ipynb

### 10.3.4.- Pregunta - Respuesta

In [ ]:
# from https://colab.research.google.com/github/huggingface/notebooks/blob/main/examples/question_answering.ipynb

### 10.3.5.- Generación de texto

In [3]:
# from https://colab.research.google.com/github/huggingface/blog/blob/main/notebooks/02_how_to_generate.ipynb#scrollTo=ue2kOQhXTAMU

### 10.4.1.- LangChain

In [ ]:
from langchain import PromptTemplate, HuggingFaceHub, LLMChain, OpenAI, AzureOpenAI
import os

In [ ]:
template = """Question: {question}

Answer: """
prompt = PromptTemplate(
        template=template,
    input_variables=['question']
)

# user question
question = "Which NFL team won the Super Bowl in the 2010 season?"

In [ ]:
# HUGGING FACE
os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'HF_API_KEY'

In [ ]:
# initialize Hub LLM
hub_llm = HuggingFaceHub(
        repo_id='google/flan-t5-xl',
    model_kwargs={'temperature':1e-10}
)

# create prompt template > LLM chain
llm_chain = LLMChain(
    prompt=prompt,
    llm=hub_llm
)

# ask the user question about NFL 2010
print(llm_chain.run(question))

In [ ]:
qs = [
    {'question': "Which NFL team won the Super Bowl in the 2010 season?"},
    {'question': "If I am 6 ft 4 inches, how tall am I in centimeters?"},
    {'question': "Who was the 12th person on the moon?"},
    {'question': "How many eyes does a blade of grass have?"}
]
res = llm_chain.generate(qs)
res

In [ ]:
multi_template = """Answer the following questions one at a time.

Questions:
{questions}

Answers:
"""
long_prompt = PromptTemplate(template=multi_template, input_variables=["questions"])

llm_chain = LLMChain(
    prompt=long_prompt,
    llm=flan_t5
)

qs_str = (
    "Which NFL team won the Super Bowl in the 2010 season?\n" +
    "If I am 6 ft 4 inches, how tall am I in centimeters?\n" +
    "Who was the 12th person on the moon?" +
    "How many eyes does a blade of grass have?"
)

print(llm_chain.run(qs_str))

In [ ]:
# OPENAI
os.environ['OPENAI_API_TOKEN'] = 'OPENAI_API_KEY'

In [ ]:
davinci = OpenAI(model_name='text-davinci-003')

llm = AzureOpenAI(
    deployment_name="your-azure-deployment", 
    model_name="text-davinci-003"
)

In [ ]:
llm_chain = LLMChain(
    prompt=prompt,
    llm=davinci
)

print(llm_chain.run(question))

In [ ]:
qs = [
    {'question': "Which NFL team won the Super Bowl in the 2010 season?"},
    {'question': "If I am 6 ft 4 inches, how tall am I in centimeters?"},
    {'question': "Who was the 12th person on the moon?"},
    {'question': "How many eyes does a blade of grass have?"}
]
llm_chain.generate(qs)

In [ ]:
llm_chain = LLMChain(
    prompt=long_prompt,
    llm=davinci
)

qs_str = (
    "Which NFL team won the Super Bowl in the 2010 season?\n" +
    "If I am 6 ft 4 inches, how tall am I in centimeters?\n" +
    "Who was the 12th person on the moon?" +
    "How many eyes does a blade of grass have?"
)

print(llm_chain.run(qs_str))

### 10.4.2.- Auto-GPT

In [1]:
# from https://github.com/IntelligenzaArtificiale/Free-Auto-GPT/blob/main/README.md

In [2]:
!pip install diffusers transformers accelerate

  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.13.1
    Uninstalling huggingface-hub-0.13.1:
      Successfully uninstalled huggingface-hub-0.13.1


In [3]:
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video

pipe = DiffusionPipeline.from_pretrained("damo-vilab/text-to-video-ms-1.7b", torch_dtype=torch.float16, variant="fp16")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()

prompt = "Spiderman is surfing"
video_frames = pipe(prompt, num_inference_steps=25).frames
video_path = export_to_video(video_frames)

C:\ProgramData\Anaconda3\lib\site-packages\huggingface_hub\file_download.py:133: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Anna\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
text_encoder\model.fp16.safetensors not found


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>                                                                                      │
│                                                                                                  │
│    7 pipe.enable_model_cpu_offload()                                                             │
│    8                                                                                             │
│    9 prompt = "Spiderman is surfing"                                                             │
│ ❱ 10 video_frames = pipe(prompt, num_inference_steps=25).frames                                  │
│   11 video_path = export_to_video(video_frames)                                                  │
│   12                                                                                             │
│                                                                                                  │
│ C:\ProgramData\Anaconda3\lib\site-packages\torch\autograd\grad_mode.py:27 in decorate_context    │
│                                                                                                  │
│    24 │   │   @functools.wraps(func)                                                             │
│    25 │   │   def decorate_context(*args, **kwargs):                                             │
│    26 │   │   │   with self.clone():                                                             │
│ ❱  27 │   │   │   │   return func(*args, **kwargs)                                               │
│    28 │   │   return cast(F, decorate_context)                                                   │
│    29 │                                                                                          │
│    30 │   def _wrap_generator(self, func):                                                       │
│                                                                                                  │
│ C:\ProgramData\Anaconda3\lib\site-packages\diffusers\pipelines\text_to_video_synthesis\pipeline_ │
│ text_to_video_synth.py:634 in __call__                                                           │
│                                                                                                  │
│   631 │   │   │   │   latent_model_input = self.scheduler.scale_model_input(latent_model_input   │
│   632 │   │   │   │                                                                              │
│   633 │   │   │   │   # predict the noise residual                                               │
│ ❱ 634 │   │   │   │   noise_pred = self.unet(                                                    │
│   635 │   │   │   │   │   latent_model_input,                                                    │
│   636 │   │   │   │   │   t,                                                                     │
│   637 │   │   │   │   │   encoder_hidden_states=prompt_embeds,                                   │
│                                                                                                  │
│ C:\ProgramData\Anaconda3\lib\site-packages\torch\nn\modules\module.py:1110 in _call_impl         │
│                                                                                                  │
│   1107 │   │   # this function, and just call forward.                                           │
│   1108 │   │   if not (self._backward_hooks or self._forward_hooks or self._forward_pre_hooks o  │
│   1109 │   │   │   │   or _global_forward_hooks or _global_forward_pre_hooks):                   │
│ ❱ 1110 │   │   │   return forward_call(*input, **kwargs)                                         │
│   1111 │   │   # Do not call functions when jit is used                                          │
│   1112 │   │   full_backward_hooks, non_full_backward_hooks = [], []                             │
│   1113 │   │   if self._backward_hooks or _global_backward_